In [20]:
import httpx

origin = "Brooklyn, NY"
destination = "Beacon, NY"


def geocode_place(query):
    response = httpx.get(
        "https://nominatim.openstreetmap.org/search",
        params={
            "q": query,
            "format": "jsonv2",
            "limit": 1,
            "countrycodes": "us",
        },
        headers={"User-Agent": "pleasant-rides-notebook/1.0"},
        timeout=30.0,
    )
    return response.json()[0]


origin_point = geocode_place(origin)
destination_point = geocode_place(destination)

print(origin, "->", (float(origin_point["lat"]), float(origin_point["lon"])))
print(
    destination,
    "->",
    (float(destination_point["lat"]), float(destination_point["lon"])),
)

Brooklyn, NY -> (40.6526006, -73.9497211)
Beacon, NY -> (41.5048402, -73.9695409)


Uses the [route API](https://valhalla.github.io/valhalla/api/route/api-reference/) from the [demo instance](https://valhalla.openstreetmap.de/) of [Valhalla](https://valhalla.github.io/valhalla/).


In [15]:
valhalla_params = {
    "locations": [
        {"lat": float(origin_point["lat"]), "lon": float(origin_point["lon"])},
        {
            "lat": float(destination_point["lat"]),
            "lon": float(destination_point["lon"]),
        },
    ],
    "costing": "bicycle",
    "costing_options": {
        "bicycle": {
            "bicycle_type": "road",
            "use_roads": 0.2,
            "use_hills": 0.3,
            "use_ferry": 0.0,
            "avoid_bad_surfaces": 0.8,
        }
    },
    "directions_options": {"units": "miles"},
}

In [16]:
import json

import folium
import httpx
import polyline
from IPython.display import display

In [21]:
url = "https://valhalla1.openstreetmap.de/route"
request_json = json.dumps(valhalla_params, separators=(",", ":"))

response = httpx.get(url, params={"json": request_json}, timeout=60.0)

route_response = response.json()
print(f"HTTP {response.status_code}")
print("Top-level keys:", list(route_response.keys()))

HTTP 200
Top-level keys: ['trip']


In [23]:
def extract_trips(payload):
    trips = [("Primary", payload["trip"])]
    for i, alt in enumerate(payload.get("alternates", []), start=1):
        trips.append((f"Alternate {i}", alt.get("trip", alt)))
    return trips


def decode_trip_shape(trip):
    points = []
    for leg in trip["legs"]:
        leg_points = polyline.decode(leg["shape"], precision=6)
        if points and leg_points and points[-1] == leg_points[0]:
            leg_points = leg_points[1:]
        points.extend(leg_points)
    return points


trips = extract_trips(route_response)
route_points = [(label, decode_trip_shape(trip)) for label, trip in trips]

center = route_points[0][1][0]
m = folium.Map(location=[center[0], center[1]], zoom_start=14, tiles="CartoDB Positron")

colors = ["#E63946", "#1D3557", "#2A9D8F", "#F4A261", "#457B9D"]
for idx, (label, points) in enumerate(route_points):
    color = colors[idx % len(colors)]
    folium.PolyLine(
        points,
        color=color,
        weight=5,
        opacity=0.85,
        tooltip=label,
    ).add_to(m)

start = route_points[0][1][0]
end = route_points[0][1][-1]
folium.Marker(start, tooltip="Start", icon=folium.Icon(color="green")).add_to(m)
folium.Marker(end, tooltip="End", icon=folium.Icon(color="red")).add_to(m)

# fit the whole route
all_points = [pt for _, pts in route_points for pt in pts]
lats = [pt[0] for pt in all_points]
lons = [pt[1] for pt in all_points]
m.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]])

print(f"Mapped {len(route_points)} route(s).")
display(m)

Mapped 1 route(s).
